# Fine-Tune LLMs Locally with InstructLab

A practical companion to the supplied IBM course reading, prepared for PaulRobertH.

**Objectives:** define a tokenization skill, prepare a taxonomy contribution, inspect teacher-generated examples, train a student, and compare its responses with a baseline.

**How to use:** select a Python 3 kernel and run the notebook from the top. The default Python cells use only the standard library, prepare files, and check examples. They do not install InstructLab, download models, or train a model. The terminal sections describe the actual InstructLab workflow.

Allow about 15–25 minutes for preparation. Model downloads, generation, and training take additional time depending on hardware.

## Version and hardware

The original [InstructLab project](https://github.com/instructlab/instructlab) is archived. This notebook documents the historical **0.18.0** CLI, rather than assuming the reading's commands work with every release. Its documented environment is Linux or Apple Silicon macOS, Python 3.10/3.11, a C++ compiler, and roughly 60 GB free disk for the full workflow. Consult the [versioned installation guide](https://github.com/instructlab/instructlab/blob/v0.18.0/README.md) for hardware-specific setup.

Your Codespace is suitable for this notebook's preparation cells. CPU-only model work can be slow; the full 7B workflow has not been tested in your Codespace. Native Windows PowerShell is not the Linux terminal used in the instructions.

Teacher → reviewed synthetic examples → student training → held-out evaluation. A quantized GGUF used for inference is different from the trainable student model. Quantization and LoRA support depend on the training backend; do not assume every platform uses QLoRA.


In [ ]:
from pathlib import Path
import ast
import json
import os
import platform
import re
import shutil
import urllib.request

LAB = Path('instructlab_lab')
LAB.mkdir(exist_ok=True)
print('Python:', platform.python_version())
print('Operating system:', platform.system())
print('Free disk (GB):', round(shutil.disk_usage('.').free / 1024**3, 1))
print('ilab executable:', shutil.which('ilab') or 'not installed (preparation still works)')


## 1. Define a precise skill

We will split English text into words and punctuation. Keep an apostrophe inside a word: `Don't` stays one token. Preserve spelling and case. Return only a Python list of strings. This is an educational word tokenizer, not a model's subword tokenizer.

The supplied reading mixes contraction expansion and punctuation splitting. We use a single consistent rule so training examples and evaluation agree.


In [ ]:
def tokenize_reference(text):
    return re.findall(r"[A-Za-z0-9]+(?:['’][A-Za-z0-9]+)*|[^\w\s]", text)

assert tokenize_reference("Don't stop.") == ["Don't", 'stop', '.']
assert tokenize_reference('Ready, Paul?') == ['Ready', ',', 'Paul', '?']
assert tokenize_reference('') == []
print(tokenize_reference('She sells seashells by the seashore.'))


## 2. Build seed examples and taxonomy files

These original examples follow the [version 2 skill format](https://github.com/instructlab/taxonomy#skills-yaml-examples). At least five question/answer pairs are needed. Keep evaluation sentences separate from these seeds.


In [ ]:
seed_texts = [
    'Paul studies language models.',
    'The notebook runs in a browser.',
    'Can we compare two answers?',
    'Save the file, then continue.',
    "Don't change the spelling.",
    'A teacher creates new examples.',
    'Training takes time!',
    'We tested 12 short sentences.',
]
task = "Split English text into words and punctuation. Preserve case and spelling; keep internal apostrophes in words. Return only a Python list of strings."
taxonomy = {
    'version': 2,
    'task_description': task,
    'created_by': 'PaulRobertH',
    'seed_examples': [
        {'question': f'Tokenize {json.dumps(text)}', 'answer': repr(tokenize_reference(text))}
        for text in seed_texts
    ],
}
skill_dir = LAB / 'taxonomy' / 'compositional_skills' / 'linguistics' / 'tokenizer'
skill_dir.mkdir(parents=True, exist_ok=True)
# Emit conventional YAML without adding a PyYAML dependency.
lines = ['version: 2', 'task_description: ' + json.dumps(task), 'created_by: PaulRobertH', 'seed_examples:']
for example in taxonomy['seed_examples']:
    lines.extend(['  - question: ' + json.dumps(example['question']), '    answer: ' + json.dumps(example['answer'])])
qna = skill_dir / 'qna.yaml'
qna.write_text('\n'.join(lines) + '\n', encoding='utf-8')
(skill_dir / 'attribution.txt').write_text(
    'Original educational tokenization examples generated for PaulRobertH with Codex assistance.\n'
    'Workflow inspired by the user-supplied IBM InstructLab reading.\n'
    'No examples copied from the reading.\n', encoding='utf-8')
assert len(taxonomy['seed_examples']) >= 5
for text, example in zip(seed_texts, taxonomy['seed_examples']):
    assert ast.literal_eval(example['answer']) == tokenize_reference(text)
print(qna.read_text(encoding='utf-8'))


## 3. Install and initialise InstructLab (terminal)

Use a **separate** environment from your other course notebooks. Run these commands in a Linux terminal with Python 3.11 and a working compiler. The version pin fixes the CLI family, but installation compatibility still depends on the platform and dependency resolution. This installation has not been executed here.

```bash
mkdir -p instructlab-runtime
cd instructlab-runtime
python3.11 -m venv .venv
source .venv/bin/activate
python -m pip install --upgrade pip
python -m pip install 'instructlab==0.18.0'
ilab --version
ilab config init
```

During initialisation, choose the CPU profile if appropriate and note the actual taxonomy directory. Copy the generated `linguistics/tokenizer` folder into that directory's `compositional_skills/linguistics/tokenizer` folder. The notebook's `instructlab_lab/taxonomy` is a staging folder, not a clone of the official taxonomy repository.

```bash
ilab taxonomy diff
```

The notebook checks example consistency; `ilab taxonomy diff` performs the actual InstructLab taxonomy validation. Resolve any schema errors before generating data. The archived taxonomy cannot accept new upstream contributions, but you can use a local clone.


## 4. Download and chat with the base models (terminal)

Activate the InstructLab environment in each terminal. Download the default teacher and the Granite inference model:

```bash
ilab model download
ilab model download --repository instructlab/granite-7b-lab-GGUF --filename granite-7b-lab-Q4_K_M.gguf
ilab model list
```

If access requires authentication, supply `HF_TOKEN` through your environment or Codespaces secrets; do not put a token in a notebook or commit it. Model availability and access rules may have changed since the reading.

Use the paths printed by the download command. In terminal A:

```bash
ilab model serve --model-path /actual/path/to/granite-7b-lab-Q4_K_M.gguf
```

In terminal B:

```bash
ilab model chat --model /actual/path/to/granite-7b-lab-Q4_K_M.gguf
```

Ask the held-out prompts below and save the answers. Exit chat, then stop serving with Ctrl+C when finished. Generated responses are measurements, not predetermined results.


In [ ]:
held_out = ['She sells seashells by the seashore.', "We're ready, aren't we?", 'Bring 3 notebooks!']
assert not set(held_out) & set(seed_texts)
for text in held_out:
    print(f'Tokenize {json.dumps(text)}')


## 5. Generate synthetic data with the teacher (terminal)

Start the default teacher in terminal A:

```bash
ilab model serve
```

In terminal B, validate the taxonomy and request 100 instructions:

```bash
ilab data generate --help
ilab data generate --num-instructions 100
```

Keep the teacher running until generation completes. Inspect the generated dataset paths printed by the command. Review format consistency, duplicates, incorrect tokenization, and overlap with evaluation prompts before using the data for training.

The optional cell below lets you request a small sample directly from a running teacher's OpenAI-compatible endpoint. It is an inspection exercise, not a replacement for `ilab data generate`. It is disabled by default.


In [ ]:
RUN_TEACHER_SAMPLE = False
ENDPOINT = 'http://127.0.0.1:8000/v1'

if RUN_TEACHER_SAMPLE:
    with urllib.request.urlopen(ENDPOINT + '/models', timeout=30) as response:
        teacher_id = json.load(response)['data'][0]['id']
    request_body = {
        'model': teacher_id,
        'messages': [
            {'role': 'system', 'content': task},
            {'role': 'user', 'content': 'Create five new English sentences and their token lists. Return a JSON array with text and tokens keys only. Examples: ' + json.dumps(taxonomy['seed_examples'][:5])},
        ],
        'temperature': 0.3,
        'max_tokens': 800,
    }
    request = urllib.request.Request(
        ENDPOINT + '/chat/completions', data=json.dumps(request_body).encode(),
        headers={'Content-Type': 'application/json'}, method='POST')
    with urllib.request.urlopen(request, timeout=600) as response:
        result = json.load(response)
    raw = result['choices'][0]['message']['content']
    (LAB / 'teacher_sample.txt').write_text(raw, encoding='utf-8')
    print(raw)
else:
    print('Teacher request skipped. Start the teacher and set RUN_TEACHER_SAMPLE=True to request a sample.')


## 6. Review candidate examples

Paste teacher output into `candidates` as dictionaries with `text` and `tokens` fields. The provided entries are deliberately labelled fixtures, including one incorrect answer. They demonstrate validation only; they are not teacher-generated data and are not training results.


In [ ]:
candidates = [
    {'text': 'Please open the notebook.', 'tokens': ['Please', 'open', 'the', 'notebook', '.']},
    {'text': "Don't stop!", 'tokens': ['Do', 'not', 'stop', '!']},
]
accepted = []
seen = set(seed_texts + held_out)
for row in candidates:
    text = row.get('text')
    tokens = row.get('tokens')
    valid = (isinstance(text, str) and isinstance(tokens, list)
             and all(isinstance(t, str) for t in tokens)
             and tokens == tokenize_reference(text) and text not in seen)
    print('ACCEPT' if valid else 'REJECT', repr(text))
    if valid:
        accepted.append(row)
        seen.add(text)
print('Reviewed fixtures:', len(accepted), 'accepted. Replace fixtures with real teacher examples before training.')


## 7. Train the student (terminal)

Stop teacher serving/chat to free memory. Download the trainable Granite repository; the GGUF above was for inference.

```bash
ilab model download --repository instructlab/granite-7b-lab
ilab model train --help
```

Set the training model and data paths in the configuration, or supply the supported flags shown by this version's help:

```bash
ilab model train --model-path /actual/path/to/instructlab/granite-7b-lab --data-path /actual/path/to/train.jsonl
```

Select a backend appropriate for your hardware using the versioned guide and `--help`. Do not use an MLX output path on Linux. Training a 7B model requires substantially more resources than running the preparation cells, and can take hours. No training is launched automatically by this notebook.

Record the actual checkpoint path, selected backend, dataset path, package versions, training duration, and parameters. Avoid committing model weights to this course repository.


## 8. Serve and evaluate the trained model

The reading's `*-mlx-q` paths describe an Apple Silicon workflow. The conversion and serving steps depend on the backend and output format. Inspect these commands before using the appropriate one:

```bash
ilab model test --help
ilab model convert --help
ilab model serve --help
```

For an output already in GGUF format:

```bash
ilab model serve --model-path /actual/path/to/trained-model.gguf
```

Open another terminal and chat with that same model path. Ask exactly the same held-out prompts used for the base model. Where supported, keep decoding settings the same. Fine-tuning may improve results but improvement must be measured.


In [ ]:
def score_answer(text, answer):
    try:
        parsed = ast.literal_eval(answer.strip())
    except (ValueError, SyntaxError):
        return {'valid_list': False, 'exact_match': False}
    valid = isinstance(parsed, list) and all(isinstance(t, str) for t in parsed)
    return {'valid_list': valid, 'exact_match': valid and parsed == tokenize_reference(text)}

assert score_answer('Hi!', "['Hi', '!']")['exact_match']
assert not score_answer('Hi!', 'Here are the tokens: Hi!')['valid_list']
assert not score_answer('Hi!', "['hi', '!']")['exact_match']

# Paste real responses from the base and trained model, in held_out order.
base_responses = []
trained_responses = []
if len(base_responses) == len(trained_responses) == len(held_out):
    report = {}
    for label, answers in [('base', base_responses), ('trained', trained_responses)]:
        rows = [score_answer(text, answer) for text, answer in zip(held_out, answers)]
        report[label] = {'examples': rows, 'exact_match_rate': sum(r['exact_match'] for r in rows) / len(rows)}
    (LAB / 'evaluation.json').write_text(json.dumps(report, indent=2), encoding='utf-8')
    print(json.dumps(report, indent=2))
else:
    print('No model comparison yet. Paste one real response per held-out prompt into each list.')


## Exercises

1. Add five new seed sentences while retaining the same tokenization rule.
2. Generate and review real teacher examples. Describe two rejected examples and why they failed.
3. Compare base and trained model exact-match rates on at least ten unseen sentences.
4. Change the skill to return JSON arrays. Update the seeds, task description, and evaluator together.

## Troubleshooting

- **`ilab` not found:** activate the separate InstructLab environment in that terminal.
- **Python 3.12+ installation failures:** this historical release documents Python 3.10/3.11.
- **Taxonomy not detected:** copy the staging skill into the configured taxonomy clone and run `ilab taxonomy diff`.
- **Connection refused:** start the teacher server and check its endpoint/port.
- **Unknown CLI option:** verify `ilab --version` and that command's `--help`; do not mix versions.
- **Out of memory or very slow training:** use hardware suitable for the selected backend; finish the preparation cells in Codespaces first.

## Sources and provenance

Adapted from the user-supplied “Fine-Tune LLMs Locally with InstructLab” reading. Prose and seed examples in this companion are newly written, rather than a verbatim copy. Historical CLI commands were checked against the [0.18.0 README](https://github.com/instructlab/instructlab/blob/v0.18.0/README.md). See the [taxonomy specification](https://github.com/instructlab/taxonomy) and [current project announcement](https://github.com/instructlab/instructlab) for context. The project points to SDG Hub and Training Hub as successor components; this notebook retains the course's historical InstructLab workflow.

Validation covers the standard-library cells, seed consistency, and response scoring. InstructLab installation, model availability, generation, and full model training were not executed during notebook creation.
